# NER with CNN-BiLSTM

Giuseppe Termerissa


## Setup


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import os
import json
import numpy as np
import glob
import shutil
import tensorflow

import sys

sys.path.insert(0, "./src/")

### Download dependencies

This is unnecessary unless the notebook is being ran on Colab with no pre-downloaded datasets, and will be normally skipped.


In [ ]:
try:
    import google.colab  # type: ignore

    IN_COLAB = True
except Exception:
    IN_COLAB = False

In [ ]:
import os

if "glove.6B" not in os.listdir("../data"):
    !wget "https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip"

    # four file with different numbers of parameters: 50, 100, 200, 300
    !unzip glove.6B.zip -d ../data/glove.6B/

## Pipeline


In [ ]:
hyper = {
    # CNN hyperameters
    "CONV_WIDTH": 4,
    "FILTERS": 30,
    # BiLSTM hyperparameters
    "LSTM_STATE_SIZE": 300,
    "LSTM_DROPOUT": 0.68,
    "REC_DROPOUT": 0.25,
    # Model hyperparameters
    "WORD_EMB_DIR": "glove.6B/glove.6B.100d.txt",
    "EPOCHS": 80,
    "BATCH_SIZE": 128,
    "OPTIMIZER": "rmsprop",
}

In [ ]:
from src.datasets import load_conll2003

conll2012_dir = "../data/conll-2012"
conll2003_dir = "../data/conll003-englishversion"

train_sentences, test_sentences = load_conll2003(conll2003_dir)

len(train_sentences), len(test_sentences)

In [ ]:
from src.pipeline import (
    unpack_sentences,
    build_part_of_phrase_lu,
    build_labels_lu,
    caps_lu,
    load_word_embeddings,
    create_char_lu,
)

labels, vocab, phrase_parts = unpack_sentences([train_sentences, test_sentences])

phrase_lu, phrase_emb = build_part_of_phrase_lu(phrase_parts)

labels_lu = build_labels_lu(labels)

case_emb = np.identity(len(caps_lu), dtype="float32")

words_lu, word_emb = load_word_embeddings(f"../data/{hyper['WORD_EMB_DIR']}", vocab)

char_lu = create_char_lu()

In [ ]:
from src.pipeline import sentences_to_matrix


train_set = sentences_to_matrix(
    train_sentences, caps_lu, words_lu, char_lu, phrase_lu, labels_lu
)
test_set = sentences_to_matrix(
    test_sentences, caps_lu, words_lu, char_lu, phrase_lu, labels_lu
)

## Model Setup


In [ ]:
from src.model import create_model


model = create_model(
    hyper,
    word_emb,
    case_emb,
    phrase_emb,
    char_lu,
    labels_lu,
)

## Training


In [ ]:
from src.pipeline import pad_dataset

tokens, caps, chars, labels, phrasetags = pad_dataset(train_set)

X = [tokens, caps, chars, phrasetags]
y = labels

history = model.fit(
    X,
    y,
    epochs=hyper["EPOCHS"],
    batch_size=hyper["BATCH_SIZE"],
)

## Evaluation


In [ ]:
# compute predictions on the test set
from src.evaluation import predict_dataset

y_true, y_pred = predict_dataset(test_set, model)

# reverse label lookup
label_idx_lu = {v: k for k, v in labels_lu.items()}

# convert back to labels
pred_label = []
for sentence in y_pred:
    pred_label.append([label_idx_lu[element] for element in sentence])

true_label = []
for sentence in y_true:
    true_label.append([label_idx_lu[element] for element in sentence])

In [ ]:
from src.evaluation import evaluate_pred

pre, rec, f1 = evaluate_pred(pred_label, true_label)

print(f"Precision:  {pre:.3f}")
print(f"Recall:     {rec:.3f}")
print(f"F1-Score:   {f1:.3f}")